In [ ]:
import glob
import os

from matplotlib import font_manager as fm
import matplotlib as mpl

font_dir = os.path.expanduser("~/.local/share/fonts")
for path in glob.glob(os.path.join(font_dir, "lm*.otf")):
    fm.fontManager.addfont(path)
for path in glob.glob(os.path.join(font_dir, "cmu*.otf")):
    fm.fontManager.addfont(path)

mpl.rcParams.update({
    "text.usetex": False,
    "mathtext.fontset": "cm",
    "axes.unicode_minus": False,

    "font.family": "serif",

    "font.serif": ["LMRoman10", "CMU Serif", "DejaVu Serif"],
    "font.sans-serif": ["LMSans10", "CMU Sans Serif", "DejaVu Sans"],
    "font.monospace": ["LMMono10", "CMU Typewriter Text", "DejaVu Sans Mono"],
})


In [ ]:
%matplotlib inline

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import xarray as xr

NC = Path("/home/jpalmo/fs09/Projects/TROPOMI_fire/paper/data/tropomi_williams_20190803.nc")
CO_PLUME = 3e18

ds = xr.open_dataset(NC)
co = ds["CO_column_number_density"]


def as_molec(da):
    return da * (1e15 if float(np.nanmedian(da)) < 1e6 else 1.0)


co = as_molec(co)
plume = co > CO_PLUME

plume_HCHO = ds.where(plume)["tropospheric_HCHO_column_number_density"].values.flatten()
background_HCHO = ds.where(~plume)["tropospheric_HCHO_column_number_density"].values.flatten()
plume_NO2 = ds.where(plume)["tropospheric_NO2_column_number_density"].values.flatten()
background_NO2 = ds.where(~plume)["tropospheric_NO2_column_number_density"].values.flatten()

fig, axs = plt.subplots(1, 2, figsize=(10, 5))

axs[0].hist(plume_HCHO / 1e15, bins=np.arange(0, 25, 1), alpha=0.5, label="Plume", color="tab:red", density=True)
axs[0].hist(background_HCHO / 1e15, bins=np.arange(0, 25, 1), alpha=0.5, label="Background", color="tab:blue", density=True)
axs[0].legend()
axs[0].set_xlabel(r"HCHO Column Density [1e15 molec/cm$^2$]")
axs[0].set_ylabel(r"Density")
axs[1].hist(plume_NO2 / 1e15, bins=np.arange(0, 5, 0.2), alpha=0.5, label="Plume", color="tab:red", density=True)
axs[1].hist(background_NO2 / 1e15, bins=np.arange(0, 5, 0.2), alpha=0.5, label="Background", color="tab:blue", density=True)
axs[1].legend()
axs[1].set_xlabel(r"NO$_2$ Column Density [1e15 molec/cm$^2$]")

png_path = "/home/jpalmo/fs09/Projects/TROPOMI_fire/paper/figs/pngs/figs06.png"
plt.savefig(png_path, dpi=300)
pdf_path = "/home/jpalmo/fs09/Projects/TROPOMI_fire/paper/figs/pdfs/figs06.pdf"
plt.savefig(pdf_path, dpi=300)
plt.show()
